In [1]:
from datascience import *
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')
import warnings
warnings.simplefilter(action="ignore", category=FutureWarning)


In [2]:
def to_standard_units(x_given):
    return (x_given - np.mean(x_given))/np.std(x_given)

def from_standard_units(x_standard, mean, std):
    return std*x + mean
    
def make_correlated_data(r):
    x = np.random.normal(0, 1, 179)
    z = np.random.normal(0,1,179)
    y = r*x + np.sqrt(1-r**2)*z
    return x,y

def make_r_table(r):
   #  np.random.seed(8)
    x,y = make_correlated_data(r)
    x = standard_units(x)
    y = standard_units(y)
    return Table().with_columns(f'x\n r = {r}',x,'y',y)

def draw_r_table(r):
    my_table = make_r_table(r)
    my_table.scatter('x','y')

def calculate_r_from_table(tbl, x_column_name, y_column_name):
    x_standard = to_standard_units(tbl.column(x_column_name))
    y_standard = to_standard_units(tbl.column(y_column_name))
    return np.average(x_standard * y_standard)

def calculate_r_from_arrays(x_array, y_array):
    x_standard = to_standard_units(x_array)
    y_standard = to_standard_units(y_array)
    return np.average(x_standard * y_standard)                  

def regression_line_in_given_units(x_array, y_array):
    r = calculate_r_from_arrays(x_array, y_array)
    m = r*np.std(y_array)/np.std(x_array)
    b = np.average(y_array) - m*np.average(x_array)
    return m,b
    
def apply_regression_line_to_table(tbl,x_column_name,y_column_name):
    x_array = tbl.column(x_column_name)
    y_array = tbl.column(y_column_name)
    m,b = regression_line_in_given_units(x_array,y_array)
    predictions = m*x_array + b
    return predictions
    
def draw_line(slope=0, intercept=0,x=make_array(-4,4), color = 'r'):
    y = x*slope + intercept
    plots.plot(x,y,color=color)
    
def draw_regression_graph(r):
    tbl = make_r_table(r)
    tbl.scatter('x')
    draw_line(1)
    draw_line(r,color='g')
    

In [3]:
sales_regression = Table.read_table('houses.csv')
# Add column of 1s:
ones = [1]*sales_regression.num_rows
sales_regression = sales_regression.with_column('One', ones)


sales_nn = Table.read_table('houses_reduced.csv')

In [4]:
# Split into training set and testing set
sales_regression = sales_regression.sample()
sles_nn = sales_nn.sample()

sales_regression_training, sales_regression_test = sales_regression.split(int(sales_regression.num_rows/2))
sales_nn_training, sales_nn_test = sales_nn.split(int(sales_nn.num_rows/2))

In [5]:
# regression predictions functions
def make_prediction_for_row(slopes, row):
    row_as_array = np.array(row[1:]) # remove sale price
    return sum(slopes * row_as_array)

def root_mean_square_error(slopes):
    predictions = make_array()
    for i in np.arange(sales_regression_training.num_rows):
        prediction = make_prediction_for_row(slopes, sales_regression_training.row(i))
        predictions = np.append(predictions, prediction)
    sales_array = sales_regression_training.column('SalePrice')
    return np.mean((sales_array - predictions)**2)**0.5


best_slopes = minimize(root_mean_square_error,start=[1]*(sales_regression_training.num_columns-1),smooth=True,array=True)

def regression_prediction(row):
    return np.round(make_prediction_for_row(best_slopes, row),-2)

In [6]:
# nearest neighbor prediction functions
def row_distance_squared(row_1, row_2):
    row_1_array = np.array(row_1)[1:]
    row_2_array = np.array(row_2)[1:]
    return np.sum((row_2_array - row_1_array)**2)

def nn_prediction(row_to_predict,number_of_neighbors=5):
    distances = make_array()
    for row in sales_nn_training.rows:
        distances = np.append(distances, row_distance_squared(row_to_predict,row))
    table_with_distances = sales_nn_training.with_column('Distance',distances).sort('Distance').take(np.arange(number_of_neighbors))
    sales_prices = table_with_distances.column('SalePrice')
    return np.average(sales_prices)

In [17]:
def root_mean_square_error_for_prediction(prediction_function, test_table):
    errors = make_array()
    for row in test_table.rows:
        error = prediction_function(row) - row.item('SalePrice')
        errors = np.append(errors, error)
    return np.sqrt(np.average(errors**2))
        

In [14]:
root_mean_square_error_for_prediction(nn_prediction, sales_nn_test)

39999.231026551555

In [15]:
root_mean_square_error_for_prediction(regression_prediction, sales_regression_test)

31609.331016547731